In [21]:
# laod secrets 
from dotenv import load_dotenv
load_dotenv()

True

In [22]:
# load the saved faiss index from the disk 

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

DB_FAISS_PATH = "vectorstore/db_faiss"

embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2") # the FAISS index only stores vectors,
#not the model that produced them. To later embed a new user question into the same vector space (so it's comparable), we need the
#identical embedding model loaded again. If you used a different model here than during ingestion, the vectors wouldn't be comparable at all

print("loading FAISS model")
db = FAISS.load_local(DB_FAISS_PATH , embedding_model , allow_dangerous_deserialization = True) # allow_dangerous_deserialization - Pickle can 
#execute arbitrary code if loaded from an untrusted source, so LangChain requires you to explicitly opt in with this flag

print("FAISS index loaded successfully")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

loading FAISS model
FAISS index loaded successfully


In [23]:
# bringing in the LLM 

import os 
from langchain_groq import ChatGroq

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
GROQ_MODEL_NAME = "llama-3.1-8b-instant"

llm = ChatGroq(
    api_key = GROQ_API_KEY,
    model = GROQ_MODEL_NAME,
    temperature = 0.5, # control randomness 
    max_tokens = 512 # maximum length of the generated answer
)

print("LLM initiated successfully" , llm )

LLM initiated successfully metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x000001AFC76F1350> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AFC76F1CD0> model_name='llama-3.1-8b-instant' temperature=0.5 model_kwargs={} groq_api_key=SecretStr('**********') groq_api_base=None groq_proxy=None max_tokens=512


In [28]:
# retrieval + generation 

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser 

CUSTOM_PROMPT_TEMPLATE = '''
Use the pieces of information provided in the context to answer the user's question.
Answer in a clear, complete, and well-explained way using only the information given in the context.
Do not just copy a single sentence — synthesize the relevant details into a full answer.
If the answer is not contained in the context, say you don't know — do not make anything up.

Context: {context}
Question: {question}

Answer:
'''

prompt = ChatPromptTemplate.from_template(CUSTOM_PROMPT_TEMPLATE)

retriever = db.as_retriever(search_kwargs = {'k':3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs )

rag_chain = (
    {"context" : retriever | format_docs , "question" : RunnablePassthrough()}
    | prompt 
    | llm 
    | StrOutputParser()
)

print("RAG chain built successfully")

RAG chain built successfully


In [33]:
# testing
response = rag_chain.invoke("What is cancer?")
print(response)

According to the provided context, cancer is defined as a disease of the genes. It is a disease that occurs when a cell loses its normal growth control restraints and starts multiplying uncontrollably, resulting in an abnormal growth known as a tumor. This definition is based on the information that genes make proteins, which are the workhorses of the cells, and that cells in the body are constantly growing, dividing, and replacing themselves.
